#  Phil Job Applier Agent - Rachel Briskman & Sanila Chowdhury (Daedalus Devs)

## Team Report #1 (03/24) - Created a function that takes in user's resume and parse it to output the extracted info & uses Skyvern to see how the library works for filling in the fields of a job website

<img src="AI_AGENT_WORKFLOW.png" width="700" />

### Installs and Imports (uncomment and run if needed)
#### We left it commented so we can use "run all" in the notebook without wasting time

In [ ]:

# %pip install chromadb
# %pip install skyvern
# %pip install langchain langchain-huggingface
# %pip install python-dotenv
# %pip install browser-use playwright
# %pip install -U langchain-google-genai
# %pip install -U browser-use

# print ("Done installing")

### Import API keys from .env file
#### (Note: This is for local use. Colab imports keys differently)

In [ ]:
import os
from dotenv import load_dotenv
import platform
import sys
from langsmith import traceable


# For DEBUGGING: print the platform it's running on
IS_MAC = platform.system() == "Darwin"
print(f"Running on: {platform.system()}")


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")
langmsith_key = os.getenv("LANGSMITH_API_KEY")


os.environ["ENABLE_AUTH"] = "false"


import nest_asyncio
import asyncio

nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")


### A website we're feeding into our agent to fill in the field (this website is used for testing purpose)

In [ ]:
url = "https://demo.applitools.com/"

### Agent takes in the filepath where the user's resume is stored and we prompt the agent to extract the info in a JSON that easy for both LLM and humans to read

In [ ]:
import asyncio
import nest_asyncio
import os
from google import genai
from IPython.display import display, Markdown

nest_asyncio.apply()
client_genai = genai.Client(api_key=gemini_key)

def parse_resume(file_path):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    Caches the result so Gemini is only called once per resume.
    """
    prompt="Please parse this resume and return its contents in a clear, readable format. Use sections with headers (like First Name, Last Name, Contact, Summary, Experience, Education, Skills) and plain text or bullet points — no JSON or code blocks."
    cache_path = file_path.replace(".pdf", "_cached.json")

    if os.path.exists(cache_path):
        print(f"Using cached resume data from {cache_path}...")
        with open(cache_path, "r") as f:
            return f.read()

    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

    # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    # save to cache so we don't call Gemini again
    with open(cache_path, "w") as f:
        f.write(response.text)
    print(f"Resume parsed and cached to {cache_path}")

    return response.text

# call the function
result = parse_resume("content/test-resumes/cs/dummy_resume_1.pdf")
display(Markdown(f"<div style='font-size: 14px'>{result}</div>"))

## Plans before next team report:
* #### Need to figure out how to see the filled in website, not as a screenshot but in another tab.
* #### Follow our workflow where the user can make corrections to the extracted resume info.

## Team Report #2 (03/31) - Created human in the loop for resume info verification, set up ChromaDB, and opened a window of the job website (dummy url) and filled in a few fields.
### (Replaced Skyvern with Browser_Use). This code will not run on Colab, only locally.

<img src="AI_AGENT_WORKFLOW_REPORT_2.png" width="700" />

### Developed a human in the loop where agent takes in the extracted info and ask the user if the info is correct. Until the user is satisifed, the agent will keep updating the extracted info based on user's feedback.

In [ ]:
from IPython.display import display, Markdown

def verify_resume_info(extracted_info):
    current_info = extracted_info
    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        display(Markdown(f"<div style='font-size: 16px'>{current_info}</div>"))
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()
        print(f"Is this information correct? (y/n): {user_confirm}")

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            break
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()
            print(f"What needs to be corrected or added? Please describe: {correction}")

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}
The user wants the following correction or addition:
{correction}
Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")
    return current_info


# call the function to store the extracted info
result = parse_resume("content/test-resumes/cs/dummy_resume_1.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

### Setting up ChromaDB which will store the correct parsed info of user's resume

In [ ]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path, email):
    # use email as unique identifier so multiple resumes can be stored
    email = input("Please enter your email address (used as your unique ID): ").strip()
    doc_id = hashlib.md5(email.encode()).hexdigest()

    # Check if resume already exists
    existing = collection.get(ids=[doc_id])
    if existing["documents"]:
        print(f"⚠️ Resume already exists for {email}. Skipping upload.")
        return doc_id
    
    # Also check if same resume content already exists under a different email
    all_docs = collection.get(include=["documents", "metadatas"])
    for i, doc in enumerate(all_docs["documents"]):
        if doc == resume_text:
            existing_email = all_docs["metadatas"][i].get("email") or all_docs["ids"][i]
            print(f"⚠️ This resume already exists in ChromaDB under {existing_email}. Skipping upload.")
            return all_docs["ids"][i]

    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path, "email": email}]
    )
    print(f"✅ Resume stored! Email: {email} | ID: {doc_id}")
    return doc_id

# store the verified resume
# doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf", "jake@su.edu")
print("Resume stored successfully!")

## Plans for spring break:
* ##### Implementing a function where our agent to fill in the fields from the job website using Skyvern (we have started this process but currently experiencing errors)
* ##### Implementing a loop for our agent to handle when it comes to short-answer response
* ##### Have a prompt that will tell the LLM to generate answer based on the context given and the info it extracted from user's resume
* ##### Have a loop where as long as the answer is too vague, keep asking user for more info and implement a function for LLM to do web search (depending on the short answer question)
* ##### Have LLM generate a short answer response and fill out the field with Skyvern
* ##### IF WE FIND TIME -> Have an evalulation where user can give the LLM feedback at the end of the overall performance and store that in ChromaDB to help Phil improve

## Team Report #3 (04/14) - Resolved the EXHAUSTED API key issue by switching to OpenRouter (paid). Implemented most of the workflow and tested the agent.
### Started the evaluation portion of this project. The agent is able to open a window to the provided URL and fill in fields. 
### It successfully extracts the job title and the company name from the website. If the agent successfully fills out the entire application, it uses those fields to update the CSV file. (Works but unreliably because of the drop-down issue).


### See video-demo.mp4 at https://drive.google.com/file/d/1_tLs-5_bf-JU4mLIjEcZ-HX6wFeDz0mI/view?usp=sharing (It gets stuck on a drop down so I ended up interrupting the cell, which closed the browser.)
### (Video too big for git so I put it in Google Drive)


### Currently testing the agent with the job application at https://job-boards.greenhouse.io/attentive/jobs/4209296009

<img src="AI_AGENT_WORKFLOW_REPORT_3.png" width="700" />

### A function to write to a CSV file with details of the job application. If a CSV file does not exist, it will create one.
#### (Not called in this cell to prevent clutter in the CSV file)

In [ ]:
import csv
import os

def save_application_history(url, company_name, time_applied, job_title):
    print ("Saving application history to application_history.csv...")
    csv_path = os.path.join(os.getcwd(), "application_history.csv")

    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="") as csvfile:
        fieldnames = ["url", "company_name", "time_applied", "job_title"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({"url": url, "company_name": company_name, "time_applied": time_applied, "job_title": job_title})
    print(f"Application history updated: {company_name} - {job_title}")
    print(f"Saved to: {csv_path}")


### A function to log agent tracing
#### (Not called in this cell to prevent clutter in the CSV file)

In [ ]:
import csv
import os

def log_trace(name, url, trace):
    print("Saving agent trace to agent_traces.csv...")
    csv_path = os.path.join(os.getcwd(), "agent_traces.csv")
    file_exists = os.path.isfile(csv_path)
    with open(csv_path, mode="a", newline="") as csvfile:
        fieldnames = ["name", "url", "trace"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow({"name": name, "url": url, "trace": trace})
    print(f"✅ Agent trace logged for {name} - {url}")

### A function to log all metrics (completion, correction, etc.) after agent finishes running.
#### (Not called in this cell to prevent clutter in the agent_resports CSV file)

In [ ]:
import json
import pandas as pd

def log_agent_report(report, name, job_title, url, filled_fields, empty_fields, metrics=None):
    print("Saving agent report to agent_reports.csv...")
    csv_path = os.path.join(os.getcwd(), "agent_reports.csv")

    def format_field_list(fields):
        if isinstance(fields, str):
            try:
                fields = json.loads(fields)
            except Exception:
                fields = [f.strip() for f in fields.split(",") if f.strip()]
        if not fields:
            return "[]"
        return "[ " + " | ".join(str(f) for f in fields) + " ]"

    filled_formatted = format_field_list(filled_fields)
    empty_formatted  = format_field_list(empty_fields)

    print("\n📝 Please rate and review the agent's performance:")
    print(f"   Filled fields : {filled_formatted}")
    print(f"   Empty fields  : {empty_formatted}")
    user_rating   = input("Your rating (1-10): ").strip()
    user_feedback = input("Your feedback (press Enter to skip): ").strip()
    if not user_feedback:
        user_feedback = "No feedback provided"

    by_type = metrics.get("by_field_type", {}) if metrics else {}

    row = {
        "name":          name,
        "job_title":     job_title,
        "url":           url,
        "user_rating":   user_rating,
        "user_feedback": user_feedback,
        "total_fields":  metrics.get("total_fields", "N/A") if metrics else "N/A",
        "fill_rate":     metrics.get("fill_rate", "N/A") if metrics else "N/A",
    }

    for field_type, stats in by_type.items():
        for stat_name, value in stats.items():
            row[f"{field_type}_{stat_name}"] = value

    file_exists = os.path.isfile(csv_path)

    new_row = pd.DataFrame([row])
    if file_exists:
        df = pd.read_csv(csv_path, keep_default_na=False)
        df = pd.concat([df, new_row], ignore_index=True)
    else:
        df = new_row
    df = df.fillna("N/A")
    df.to_csv(csv_path, index=False)

    print(f"✅ Agent report logged for {name} - {job_title}")
    print(f"   Your rating: {user_rating}/10")

### Agent ask user for any additional info that's not indicated on their resume but would be helpful in filling in the fields. That info are stored in ChromaDB.
### e.g. Demographic, Work authorization.

In [ ]:
def collect_and_store_additional_info(email):
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    doc_id = hashlib.md5(email.encode()).hexdigest()

    existing = additional_collection.get(ids=[doc_id])
    existing_data = {}
    if existing["documents"]:
        for line in existing["documents"][0].split("\n"):
            if ": " in line:
                key, val = line.split(": ", 1)
                existing_data[key.strip()] = val.strip()

    print("Collecting additional demographic and work authorization info...")
    print("Enter new values or press Enter to keep existing ones:")

    def prompt(label, key, default=""):
        current = existing_data.get(key, default)
        val = input(f"{label} (current: '{current}'): ").strip()
        return val if val else current

    gender = prompt("Gender (e.g. Male, Female, Non-binary, Prefer not to say)", "Gender")
    race = prompt("Race/Ethnicity (e.g. Asian, White, Hispanic, Black, Prefer not to say)", "Race/Ethnicity")
    veteran_status = prompt("Veteran status (e.g. Not a veteran, Veteran, Prefer not to say)", "Veteran Status")
    disability_status = prompt("Disability status (e.g. No disability, Has disability, Prefer not to say)", "Disability Status")
    work_authorization = prompt("Are you authorized to work in the US? (Yes/No)", "Work Authorization")
    sponsorship = prompt("Do you require sponsorship now or in the future? (Yes/No)", "Requires Sponsorship")
    visa_type = prompt("Visa type if applicable (e.g. H1B, F1 OPT, Green Card, Citizen, N/A)", "Visa Type")

    info_text = f"""Gender: {gender or 'Prefer not to say'}
Race/Ethnicity: {race or 'Prefer not to say'}
Veteran Status: {veteran_status or 'Prefer not to say'}
Disability Status: {disability_status or 'Prefer not to say'}
Work Authorization: {work_authorization or 'Yes'}
Requires Sponsorship: {sponsorship or 'No'}
Visa Type: {visa_type or 'N/A'}""".strip()

    additional_collection.upsert(
        documents=[info_text],
        ids=[doc_id],
        metadatas=[{"type": "additional_info", "email": email}]
    )
    print(f"✅ Additional info updated in ChromaDB!")
    return info_text

def get_additional_info(email):
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    doc_id = hashlib.md5(email.encode()).hexdigest()
    results = additional_collection.get(ids=[doc_id])
    if results["documents"]:
        print(f"✅ Additional info found for {email}")
        return results["documents"][0]
    print(f"❌ No additional info found for {email}")
    return None

### Prints the additional user info stored in ChromaDB (used for testing purpose)

In [ ]:
def print_additional_info():
    print("Displaying your additional demographic and work authorization info...")
    additional_collection = chroma_client.get_or_create_collection(name="user_additional_info")
    
    email = input("Enter your email: ").strip().lower()
    if not email or email == "":
        print("No email entered. Exiting.")
        return
    
    doc_id = hashlib.md5(email.encode()).hexdigest()
    
    results = additional_collection.get(ids=[doc_id])
    
    if not results["documents"]:
        print(f"❌ No additional info found for {email}.")
        return None
    
    info = results["documents"][0]
    print("\n" + "="*40)
    print(f"📋 YOUR STORED ADDITIONAL INFO: {email}")
    print("="*40)
    print(info)
    print("="*40)
    return info
_ = print_additional_info()

### A tracing function (used later when agent object is defined)

In [ ]:
agent_trace_log = []

def my_tracer(*args):
    trace_entry = {
        "step": len(agent_trace_log) + 1,
        "thought": "No thought captured",
        "actions": [],
    }

    for arg in args:
        # The AgentOutput Pydantic model — has 'thinking' and 'action' directly on it
        if hasattr(arg, 'thinking') and hasattr(arg, 'action'):
            thought = arg.thinking or "No thought captured"
            trace_entry["thought"] = thought

            actions = arg.action if arg.action else []
            trace_entry["actions"] = [str(a) for a in actions]
            break  # ✅ found what we need, stop checking other args

        # Fallback: wrapped inside model_output
        elif hasattr(arg, 'model_output') and arg.model_output:
            mo = arg.model_output
            trace_entry["thought"] = getattr(mo, 'thinking', None) or getattr(mo, 'thought', None) or "No thought captured"
            actions = getattr(mo, 'action', []) or []
            trace_entry["actions"] = [str(a) for a in actions]
            break

    agent_trace_log.append(trace_entry)

    print(f"\n[CUSTOM TRACE] Step {trace_entry['step']}")
    print(f"  🧠 THOUGHT: {trace_entry['thought']}")
    print(f"  🎯 ACTIONS: {trace_entry['actions']}")

### This function is used to print Gemini's thoughts - a way to see how our agent is behaving

In [ ]:
from langsmith import traceable
import openai

@traceable(name="stream_gemini", run_type="llm", metadata={"model": "gemini-2.5-flash-lite"})
def stream_gemini(prompt: str) -> str:
    import openai
    client = openai.OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_KEY"),
    )
    response = client.chat.completions.create(
        model="google/gemini-2.5-flash-lite",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    full_response = ""
    for chunk in response:
        text = chunk.choices[0].delta.content or ""
        if text:
            print(text, end="", flush=True)
            full_response += text
    print()
    return full_response.strip()

### Evalulating the correctness the agent fills out the job application

In [ ]:
from langsmith import traceable
import re

@traceable(name="evaluate_field_metrics", run_type="chain", metadata={"step": "eval"})
def evaluate_field_metrics(
    filled_fields: list,
    empty_fields: list,
    skipped_uploads,
    resume_json: dict,
    rubric: str = "",
    additional_info: str = "",
) -> dict:
    FIELD_TYPES = ["dropdown", "checkbox", "calendar", "multiple_choice", "text_field", "short_answer"]
    
    skipped_uploads = skipped_uploads or []
    
    # Uploads are skipped intentionally — exclude from fill rate calculation
    fillable_fields = filled_fields + empty_fields
    fillable_total  = len(fillable_fields)
    
    if not fillable_total:
        return {}

    def get_type(f):
        f_lower = f.lower()
        if any(k in f_lower for k in ["dropdown", "ethnicity", "country", "gender", "race"]):
            return "dropdown"
        if any(k in f_lower for k in ["checkbox", "fields of study", "interests"]):
            return "checkbox"
        if any(k in f_lower for k in ["date", "calendar"]):
            return "calendar"
        if any(k in f_lower for k in ["tell us", "describe", "why", "essay", "cover letter"]):
            return "short_answer"
        return "text_field"

    def is_correct_batch(fields: list, resume_text: str, additional_info: str = "") -> dict:
        if not fields:
            return {}
        prompt = (
            f"Resume: \n{resume_text}\n\n"
            f"Additional info: \n{additional_info}\n\n"
            f"For each field below, reply YES if the field is filled in correctly, NO otherwise. "
            f"Compare the filled values with the resume and additional info.\n"
            f"Return ONLY a JSON like:\n"
            f'{{"First Name": {{"verdict": "YES", "value": "John", "reason": ""}}, '
            f'"Email": {{"verdict": "NO", "value": "wrong@email.com", "reason": "Does not match resume email"}}}}\n'
            f"Fields: {fields}"
        )
        result = stream_gemini(prompt)
        try:
            return json.loads(re.search(r'\{.*\}', result, re.DOTALL).group())
        except Exception:
            return {f: {"verdict": "NO", "value": "unknown", "reason": "Could not parse response"} for f in fields}

    correctness_map = is_correct_batch(filled_fields, resume_json.get("text", ""), additional_info)

    stats = {ft: {"total": 0, "filled": 0, "correct": 0} for ft in FIELD_TYPES}

    incorrect = {
    field: data for field, data in correctness_map.items()
    if "NO" in data.get("verdict", "NO").upper()
}
    if incorrect:
        print("\n❌ Incorrectly filled fields:")
        for field, data in incorrect.items():
            print(f"  • {field}")
            print(f"    Value : {data.get('value', 'unknown')}")
            print(f"    Reason: {data.get('reason', 'no reason given')}")
    else:
        print("\n✅ All filled fields are correct.")

    for f in fillable_fields:
        ft = get_type(f)
        stats[ft]["total"] += 1
        if f in filled_fields:
            stats[ft]["filled"] += 1
            if "YES" in correctness_map.get(f, {}).get("verdict", "NO").upper():
                stats[ft]["correct"] += 1

    by_field_type = {}
    for ft, s in stats.items():
        if s["total"] == 0:
            continue
        by_field_type[ft] = {
            "fill_rate":      f"{round(s['filled'] / s['total'] * 100, 1)}%",
            "fill_amount":    str(s["filled"]),
            "correct_rate":   f"{round(s['correct'] / s['filled'] * 100, 1)}%" if s["filled"] else "N/A",
            "correct_amount": str(s["correct"]),
        }

    metrics = {
        "total_fields":    fillable_total,
        "skipped_uploads": len(skipped_uploads),
        "fill_rate":       f"{round(len(filled_fields) / fillable_total * 100, 1)}%",
        "by_field_type":   by_field_type,
    }

    accounted_for = sum(s["total"] for s in stats.values())
    if accounted_for != fillable_total:
        print(f"⚠️ TRACKING BUG: fillable_total={fillable_total} but stats sum={accounted_for}")

    print(f"\n📊 Field Metrics:")
    print(json.dumps(metrics, indent=2))

    print("\n🔍 Field type breakdown:")
    for i, f in enumerate(fillable_fields, start=1):
        print(f"  {i:2}. {get_type(f):20s} ← '{f}'")

    return metrics


### Agent handling short-answer fields

In [ ]:
from langsmith import traceable
@traceable(name="short_answer_field", run_type="chain", metadata={"step": "short_answer"})
def short_answer_field(field_name: str, resume_text: str, job_url: str) -> str:
    print(f"🤔 Generating answer from resume...")
    answer = stream_gemini(
        f"Based on this resume, generate a concise answer for the job application field: '{field_name}'.\n"
        f"Resume:\n{resume_text}\n\n"
        f"If the resume does not have enough info to answer this field, respond with exactly: NEEDS_MORE_INFO\n"
        f"Otherwise respond with only the answer, nothing else."
    )

    if "NEEDS_MORE_INFO" in answer.upper():
        print(f"⚠️ Resume doesn't have enough info for '{field_name}'.")
        user_info = input(f"Please provide info for '{field_name}' (or press Enter to skip): ").strip()
        if not user_info:
            print(f"⏭️ Skipping '{field_name}'")
            return None

        print(f"🤔 Checking if answer is too vague...")
        vague = stream_gemini(
            f"Is this answer too vague to fill in a job application field called '{field_name}'? "
            f"Answer only YES or NO.\nAnswer: {user_info}"
        )
        while "YES" in vague.upper():
            print("⚠️ That answer is too vague.")
            user_info = input(f"Please provide a more specific answer for '{field_name}': ").strip()
            if not user_info:
                return None
            print(f"🤔 Re-checking vagueness...")
            vague = stream_gemini(
                f"Is this answer too vague to fill in a job application field called '{field_name}'? "
                f"Answer only YES or NO.\nAnswer: {user_info}"
            )

        print(f"🤔 Checking if company research is needed...")
        research_needed = stream_gemini(
            f"Does answering the job application field '{field_name}' require researching the company at {job_url}? "
            f"Answer only YES or NO."
        )
        extra_context = ""
        if "YES" in research_needed.upper():
            print(f"🔍 Researching company for '{field_name}'...")
            extra_context = stream_gemini(
                f"Summarize relevant info about the company at {job_url} "
                f"that would help answer the job application field: '{field_name}'."
            )
            print(f"✅ Company research done.")

        print(f"🤔 Generating final answer...")
        answer = stream_gemini(
            f"Generate a concise job application answer for the field '{field_name}'.\n"
            f"Resume:\n{resume_text}\n"
            f"User provided: {user_info}\n"
            f"Extra context: {extra_context}\n"
            f"Return only the answer, nothing else."
        )

    return answer

### Tracing the LLM, seeing the agent's thought as it go through the job application and filling out the field by setting browser-use's logging level to WARNING

In [ ]:
import logging

# This will show the Agent's "Thought", "Action", and "Observation" for every step
# Good for tracing.
import json
import re
from datetime import datetime
from browser_use import Agent, BrowserProfile
from browser_use.browser import BrowserSession
from browser_use.llm import ChatOpenAI

browser_logger = logging.getLogger('browser_use')

# Set browser-use logging level to INFO to see agent's Thought, Action, and Observation
logging.getLogger('browser_use').setLevel(logging.WARNING)

# print out the current logging level. (should be INFO)
print(f"Updated browser-use logger level: {logging.getLevelName(logging.getLogger('browser_use').getEffectiveLevel())}")

### Agent fills in the application field

In [29]:
import logging
import json
import re
import os
from datetime import datetime
from browser_use import Agent, BrowserProfile
from browser_use.browser import BrowserSession
from browser_use.llm import ChatOpenAI

# LangSmith tracing setup
from langsmith import traceable, Client
from langsmith.run_helpers import get_current_run_tree
from opentelemetry import metrics

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = "phil-job-applier"  # your project name in LangSmith

def get_llm():
    return ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_KEY"),
        model="gemini-2.5-flash-lite",
    )


@traceable(name="open_browser_to_job_url", run_type="chain")
async def run_open_agent(job_url: str, llm, browser_session):
    agent = Agent(
        task=f"Go to {job_url} and wait. Do not fill anything.",
        llm=llm,
        browser_session=browser_session,
        max_steps=3,
        use_vision=False,
        register_new_step_callback=my_tracer 
    )
    await agent.run()
    print(f"Opened browser to {job_url}")


@traceable(name="detect_page_fields", run_type="chain", metadata={"step": "detection"})
async def run_detect_agent(job_url: str, additional_info: str, llm, browser_session) -> dict:
    agent = Agent(
        task=(
            f"IMPORTANT - USE THIS INFO FOR DEMOGRAPHIC AND AUTHORIZATION FIELDS. IF YOU ARE UNSURE OF WHAT TO DO, SIMPLE SKIP THE FIELD (mention it to the user) AND DO NOT GO BACK TO THAT SKIPPED FIELD (no matter what):\n"
            "IF ANY FIELD IS A DROPDOWN: IGNORE IT COMPLETELY. Regardless of what information it's asking. IGNORE ALL DROPDOWN FIELDS, EVEN IF THEY ARE DEMOGRAPHIC OR AUTHORIZATION FIELDS.\n"
            f"{additional_info}\n\n"
            f"The page {job_url} is already open. "
            "Look at the page and do the following:\n"
            "1. Find the company name and job title from the page.\n"
            "2. Find any SHORT ANSWER or ESSAY fields (e.g. 'Why do you want to work here?', 'Tell us about yourself', 'Cover letter', etc.).\n"
            "Return ONLY a JSON like this:\n"
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": ["field1", "field2"]}'
            "\nIf there are no short answer fields, return: "
            '{"company_name": "Google", "job_title": "Software Engineer", "short_answer_fields": []}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_steps=5,
        max_actions_per_step=3,
        register_new_step_callback=my_tracer 
        
    )
    detect_result = await agent.run()
    raw = detect_result.final_result()

    try:
        json_match = re.search(r'\{.*\}', raw, re.DOTALL)
        detected = json.loads(json_match.group()) if json_match else json.loads(raw)
    except Exception as e:
        print(f"⚠️ Could not parse detect result: {e}")
        detected = {"company_name": "Unknown", "job_title": "Unknown", "short_answer_fields": []}

    print(f"🏢 Company: {detected.get('company_name')} | 💼 Job Title: {detected.get('job_title')}")
    return detected


@traceable(name="generate_short_answer", run_type="llm")
def generate_short_answer(field: str, resume_text: str, job_url: str) -> str:
    # calls your existing short_answer_field() function
    return short_answer_field(field, resume_text, job_url)


@traceable(name="fill_application_fields", run_type="chain", metadata={"step": "form_fill"})
async def run_fill_agent(
    job_url: str,
    resume_text: str,
    additional_info: str,
    short_answer_context: str,
    llm,
    browser_session,
    
) -> str:
    agent = Agent(
        task=(
            f"IMPORTANT - USE THIS INFO FOR DEMOGRAPHIC AND AUTHORIZATION FIELDS (for any that appear on the page):\n"
            f"{additional_info}\n\n"
            f"The page {job_url} is already open. "
            f"Use this resume to fill out the application:\n{resume_text}\n\n"
            + (f"Use these pre-generated answers for short answer fields:\n{short_answer_context}\n\n" if short_answer_context else "")
            + "Instructions:\n"
            "1. Find every empty field on the page (with the exception of file upload and dropdown fields.).\n"
            "2. If the field can be answered from the resume, fill it in. (IF THE FIELD IS NOT A DROP DOWN.)\n"
            "3. For DROPDOWN fields (any <select>, role='listbox', role='combobox', aria-haspopup='listbox'): "
                "SKIP ALL OF THEM — including country, gender, race, veteran status, and sponsorship. "
                "Do NOT click, hover, or interact. Treat them as invisible.\n"
            "4. For SENSITIVE or AUTHORIZATION fields that are NOT dropdowns (e.g. radio buttons, free text): "
                "use the additional user info above. If not found, pick the first available option.\n"
            "5. For COUNTRY fields that are NOT dropdowns: enter 'United States'.\n"
            "6. For ALL FILE UPLOAD fields: DO NOT interact with them AT ALL. Skip immediately.\n"
            "7. For SHORT ANSWER or ESSAY fields: use the pre-generated answers provided above.\n"
            "8. DO NOT click 'Submit', 'Submit application', 'Apply', or ANY button that submits the form.\n"
            "9. You are allowed to go to the next page of the application (if there is one) after filling out the current page. BUT NEVER PRESS SUBMIT.\n"
            "10. Look back at the page and review your non-dropdown fields only. If you missed a text or radio field, fix it.\n"
            "11. Your FINAL AND ONLY action must be to call the `done` tool. "
            "The text parameter of `done` must be ONLY this JSON, nothing else — no explanation, no prose:\n"
            '{"applicant_name": "NAME", "filled": ["field1", "field2"], "empty": ["field3"], "skipped_uploads": ["file1"], "company_name": "Name", "job_title": "Title"}'
        ),
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_failures=3,
        max_steps=50,
        max_actions_per_step=10,
        register_new_step_callback=my_tracer,
        include_attributes=[
            "id", "name", "type", "placeholder", "aria-label",
            "value", "required", "href", "role", "data-testid",
            "label", "class", "aria-expanded", "aria-haspopup",
        ],
    )
    result = await agent.run()
    return result.final_result()

@traceable(name="fill_application", run_type="chain")
async def fill_application(job_url: str, resume_text: str, additional_info: str = ""):
    print("Starting fill_application()\n")
    llm = get_llm()

    browser_session = BrowserSession(
        browser_profile=BrowserProfile(headless=False, keep_alive=True)
    )
    await browser_session.start()

    # Step 1: open browser
    await run_open_agent(job_url, llm, browser_session)

    # Step 2: wait for sign-in
    signed_in = input("Are you signed in (if needed)? (y/n): ").strip().lower()
    while signed_in not in ("y", "yes"):
        signed_in = input("Please sign in and then enter y to continue: ").strip().lower()

    input("Please upload your resume manually in the browser if required, then press Enter to continue...")
    print("Continuing...")

    # Step 3: detect fields
    detected = await run_detect_agent(job_url, additional_info, llm, browser_session)
    short_answer_fields = detected.get("short_answer_fields", [])
    detected_company    = detected.get("company_name", "Unknown")
    detected_job_title  = detected.get("job_title", "Unknown")

    # Step 4: generate short answers
    short_answer_context = ""
    if short_answer_fields:
        print(f"\n📋 Short answer fields found: {short_answer_fields}")
        answers = {}
        for field in short_answer_fields:
            answer = generate_short_answer(field, resume_text, job_url)
            if answer:
                answers[field] = answer
        if answers:
            short_answer_context = "\n".join([f"{k}: {v}" for k, v in answers.items()])

    print("\nStarting to fill the application form...")

    # Step 5: fill the form
    final = await run_fill_agent(
        job_url, resume_text, additional_info, short_answer_context, llm, browser_session
    )
    print("AGENT RESULT TYPE:", type(final))
    print("RAW FINAL RESULT:", repr(final))

    # Step 6: parse result
    parsed = {}
    try:
        json_match = re.search(r'\{.*\}', final, re.DOTALL)
        parsed = json.loads(json_match.group()) if json_match else json.loads(final)
    except Exception as e:
        if final and any(word in final.lower() for word in ["filled", "completed", "successfully"]):
            print("✅ Fields filled (no JSON returned by agent).")
        else:
            print(f"⚠️ Could not parse result: {e}")

    # Deduplicate and enforce mutual exclusivity between filled and empty
    filled_fields   = list(dict.fromkeys(parsed.get("filled", [])))
    empty_fields    = list(dict.fromkeys([
        f for f in parsed.get("empty", [])
        if f not in filled_fields
    ]))
    # Track these across retries so metrics are always up to date
    skipped_uploads = list(dict.fromkeys(parsed.get("skipped_uploads", [])))
    applicant_name  = parsed.get("applicant_name", "Unknown")

    print(f"📋 Filled ({len(filled_fields)}): {filled_fields}")
    if empty_fields:
        print(f"⚠️  Empty ({len(empty_fields)}): {empty_fields}")
    else:
        print("✅ All fields filled!")

    # Step 6b: feedback retry loop
    MAX_RETRIES = 2
    for attempt in range(MAX_RETRIES):
        # if not empty_fields:
        #     break

        print(f"\n🔁 Retry attempt {attempt + 1}/{MAX_RETRIES}")
        print(f"⚠️  Still empty: {empty_fields}")
        print("Tell the agent what went wrong, or press Enter to skip retrying.")
        feedback = input("Your feedback (or Enter to skip): ").strip()

        if not feedback:
            print("⏭️ Skipping retry.")
            break

        print("Which fields to retry? Press Enter to retry all empty fields.")
        print(f"Empty fields: {empty_fields}")
        fields_input  = input("Fields to retry (comma-separated, or Enter for all): ").strip()
        if not fields_input:
            print("Nothing to fix. Exiting")
            break
        
        fields_to_fix = (
            [f.strip() for f in fields_input.split(",") if f.strip()]
            if fields_input else empty_fields
        )

        print(f"\n🤖 Retrying {len(fields_to_fix)} field(s) with your feedback...")
        retry_result = await run_retry_agent(
            job_url=job_url,
            resume_text=resume_text,
            additional_info=additional_info,
            short_answer_context=short_answer_context,
            previously_filled=filled_fields,
            fields_to_fix=fields_to_fix,
            feedback=feedback,
            llm=llm,
            browser_session=browser_session,
        )

        retry_parsed = {}
        try:
            json_match   = re.search(r'\{.*\}', retry_result, re.DOTALL)
            retry_parsed = json.loads(json_match.group()) if json_match else json.loads(retry_result)
        except Exception as e:
            print(f"⚠️ Could not parse retry result: {e}")

        newly_filled  = retry_parsed.get("filled", [])
        empty_fields  = retry_parsed.get("empty", [])

        # Merge, deduplicate, and enforce mutual exclusivity after retry
        filled_fields   = list(dict.fromkeys(filled_fields + newly_filled))
        empty_fields    = list(dict.fromkeys([f for f in empty_fields if f not in filled_fields]))
        # Accumulate skipped uploads across retries too
        skipped_uploads = list(dict.fromkeys(skipped_uploads + retry_parsed.get("skipped_uploads", [])))
        # Update applicant name if retry found it and initial didn't
        if applicant_name == "Unknown":
            applicant_name = retry_parsed.get("applicant_name", "Unknown")

        print(f"✅ Retry {attempt + 1} done. Newly filled: {newly_filled}")
        if not empty_fields:
            print("✅ All fields filled after retry!")

    # Step 7: save + log (uses final filled_fields/empty_fields AFTER all retries)
    company_name = detected_company if detected_company != "Unknown" else parsed.get("company_name", "Unknown")
    job_title    = detected_job_title if detected_job_title != "Unknown" else parsed.get("job_title", "Unknown")

    save_application_history(
        url=job_url,
        company_name=company_name,
        time_applied=datetime.today().strftime("%Y-%m-%d %H:%M:%S"),
        job_title=job_title,
    )
    log_trace(name=applicant_name, url=job_url, trace=agent_trace_log)

    # Metrics calculated from final state after all retries
    metrics = evaluate_field_metrics(
        filled_fields=filled_fields,
        empty_fields=empty_fields,
        skipped_uploads=skipped_uploads,
        resume_json=json.loads(resume_text) if (isinstance(resume_text, str) and resume_text.strip().startswith("{")) else {"text": resume_text},
        rubric="Answer should be specific, professional, and relevant to the role.",
        additional_info=additional_info,
    )

    log_agent_report(
        report=final,
        name=applicant_name,
        job_title=job_title,
        url=job_url,
        filled_fields=filled_fields,
        empty_fields=empty_fields,
        metrics=metrics,
    )

    return final

In [ ]:
### Agent retries filling fields based on user feedback

@traceable(name="retry_fill_agent", run_type="chain", metadata={"step": "retry"})
async def run_retry_agent(
    job_url: str,
    resume_text: str,
    additional_info: str,
    short_answer_context: str,
    previously_filled: list,
    fields_to_fix: list,
    feedback: str,
    llm,
    browser_session,
) -> str:
    fields_to_fix_str     = "\n".join(f"  - {f}" for f in fields_to_fix)
    previously_filled_str = "\n".join(f"  - {f}" for f in previously_filled)

    task = (
        f"You already attempted to fill this job application at {job_url}.\n"
        f"These fields were already filled successfully — DO NOT touch them:\n{previously_filled_str}\n\n"
        f"These fields were missed or filled incorrectly — fix ONLY these:\n{fields_to_fix_str}\n\n"
        f"Here is the user's feedback on what went wrong:\n{feedback}\n\n"
        f"Resume:\n{resume_text}\n\n"
        + (f"Pre-generated short answers:\n{short_answer_context}\n\n" if short_answer_context else "")
        + f"Additional info (for demographic/authorization fields):\n{additional_info}\n\n"
        "Rules (same as before):\n"
        "1. Skip ALL dropdowns and file uploads.\n"
        "2. Do NOT click Submit or any button that submits the form.\n"
        "3. Only fix the fields listed above — do not re-touch already filled fields.\n"
        "4. Your FINAL AND ONLY action must be the `done` tool with ONLY this JSON:\n"
        '{"applicant_name": "NAME", "filled": ["f1", "f2"], "empty": ["f3"], "skipped_uploads": [], "company_name": "Name", "job_title": "Title"}'
    )

    agent = Agent(
        task=task,
        llm=llm,
        browser_session=browser_session,
        use_vision=True,
        max_failures=3,
        max_steps=30,
        max_actions_per_step=10,
        register_new_step_callback=my_tracer,
        include_attributes=[
            "id", "name", "type", "placeholder", "aria-label",
            "value", "required", "href", "role", "data-testid",
            "label", "class", "aria-expanded", "aria-haspopup",
        ],
    )
    return (await agent.run()).final_result()

### Main function where user uses the agent to help apply to jobs

In [32]:
import hashlib
async def apply_to_job_with_agent():
    resume_exists = input("Would you like to upload a new resume? Enter y if this is your first time (y/n): ")
    while resume_exists != "y" and resume_exists != "n" and resume_exists != "cancel":
        resume_exists = input("Invalid input. Please enter y or n: ")

    if resume_exists == "cancel":
        print("User canceled the application process. Exiting.\n")
        return

    email = input("Please enter your email address: ").strip()

    if resume_exists == "y":
        file_path = input("Please enter the path to your resume PDF: ").strip()
        extracted_info = parse_resume(file_path)
        verified_info = verify_resume_info(extracted_info)
        store_resume_in_chromadb(verified_info, file_path, email)
        resume_text = verified_info
    else:
        doc_id = hashlib.md5(email.encode()).hexdigest()
        results = collection.get(ids=[doc_id], include=["documents"])
        if not results["documents"]:
            raise RuntimeError(f"No resume found for {email}. Please upload a resume first.")
        else:
            print("Using existing resume data. Moving on to job application...")
        resume_text = results["documents"][0]
        print(f"✅ Resume fetched for {email}")

    # Collect or fetch additional info using same email
    print("Fetching additional user info from ChromaDB...")
    additional_info = get_additional_info(email)
    if not additional_info:
        print("\n📋 No additional info found. Let's collect it now.")
        additional_info = collect_and_store_additional_info(email)
    else:
        update = input("\nWould you like to update your demographic/work authorization info? (y/n): ").strip().lower()
        if update == "y":
            additional_info = collect_and_store_additional_info(email)

    url = input("Please enter the URL of the job application you are applying to: ").strip()
    if url == "":
        print("No URL entered. Exiting.")
        return
    await fill_application(url, resume_text, additional_info)

agent_trace_log = []
await apply_to_job_with_agent()

Using existing resume data. Moving on to job application...
✅ Resume fetched for jansonj@oregonstate.edu
Fetching additional user info from ChromaDB...
✅ Additional info found for jansonj@oregonstate.edu
Starting fill_application()


[CUSTOM TRACE] Step 1
  🧠 THOUGHT: The user has requested to navigate to a specific URL and wait. The previous action successfully navigated to the URL. The current browser state shows the page has loaded and contains various interactive elements. Since the user explicitly asked to 'wait' and 'do not fill anything', the next logical step is to simply wait for a specified duration to fulfill the 'wait' instruction. I will wait for 5 seconds.
  🎯 ACTIONS: ['root=WaitActionModel(wait=wait_Params(seconds=5))']

[CUSTOM TRACE] Step 2
  🧠 THOUGHT: The user requested to navigate to a specific URL and then wait. I have successfully navigated to the URL and performed the wait action in the previous step. The user explicitly stated 'Do not fill anything.' Therefore,

ERROR    [BrowserSession] ❌ Menu item with text or value 'I do not want to answer' not found



[CUSTOM TRACE] Step 27
  🧠 THOUGHT: The previous step involved clicking the 'Disability Status' dropdown. The instructions state to skip all dropdown fields and select 'I do not want to answer' for the Disability Status. The current browser state shows that the 'Disability Status' dropdown has been clicked, and the options are likely visible. The next step is to select 'I do not want to answer' from this dropdown. After this, I need to review all non-dropdown fields and then call the `done` tool.
  🎯 ACTIONS: ["root=SelectDropdownActionModel(select_dropdown=SelectDropdownOptionAction(index=26, text='I do not want to answer'))"]

[CUSTOM TRACE] Step 28
  🧠 THOUGHT: The previous action was to click the 'Disability Status' dropdown, which revealed its options. However, the attempt to select 'I do not want to answer' failed because the menu item was not found. The user request states to skip all dropdown fields and treat them as invisible. The last few steps involved clicking these dropdo

## Plans before next team report:
* #### Continue working on evaluations:
    * ##### So far, we evaluated the component level of our agent where it parsed the resume correctly, leveraging LLM calls when needed (for additional info from the user and short-answer fields), and appending the filled-out job position to a CSV file.
    * ##### At the end-to-end level, we tested how our agent handles different types of fields (right now it's unreliable on drop-down field because it sometimes get stuck and keeps choosing random or incorrect options. It even keeps going back to the same field after filling it out so we would need to change our prompt), preventing the agent from pressing submit once all fields are filled, and how accurately it fills in fields with the correct information retrieved from the resume and the user's additional input.
    * ##### Next, we need to support multi-page job sites and navigation across different job boards (Greenhouse vs. Workday), as the agent currently only works on single-page applications.


* #### Short-answer field that involves HITL:
    * ##### We still need to integrate HITL, especially when the Agent lacks info for a field. We have already implemented basic short-answer handling, but need to extend it to asking the user for more info and triggering a web search when the question requires external or company-specific knowledge (e.g., "Why do you want to work at X?"), incorporating retrieved results into the generated response.

## Team Report #4 (04/21) - Evals
### In preparation for Evals, we fixed the ChromaDB code so it now properly stores multiple resumes. (Before, it just overwrote 1 resume each time)
### We implemented tracing the LLM to see the agent's response as it fill out the application field by setting browser-use's logging level to INFO from WARNING (the cell underneath where agent is handling short-answer fields)
### Created a function test_agent_with_dummy_resumes() to easily feed the agent multiple sites and resumes.
### Made the agent create a more detailed CSV to explain what it did. Similar to `save_application_history()`. Please see `log_agent_report()` (underneath `save_application_history()`)
### For more control, one of the sites we used is a google form we made. (https://forms.gle/sAtsVUrLm416zSwM9)

### Created a function test_agent_with_dummy_resumes() to easily feed the agent multiple sites and resumes. Dummy resumes were found on https://career.oregonstate.edu/sample-resumes

<img src="AI_AGENT_WORKFLOW_REPORT_4.png" width="700" />

In [ ]:
#function takes in a list of single page urls and multi page urls.
#IMPORTANT: each resume in test_resumes/ folder should following this naming convention: dummy_resume_1.pdf  (only change the number)
import os

async def test_agent_with_dummy_resumes(resume_folder,n_resumes, single_page_urls):
    # test_resumes_dir = "/Users/sanilachowdhury/Desktop/ai_agents/job-agent/content/test-resumes/"
    test_resumes_dir = os.path.join(os.getcwd(), f"content/test-resumes/{resume_folder}/")
    if not os.path.isdir(test_resumes_dir):
        print(f"Invalid Directory: {test_resumes_dir}")
        return
    
    if not single_page_urls:
        print("No urls provided, nothing to test! Try again with some urls.\n")

    print(f"Testing {n_resumes} resumes from directory: {test_resumes_dir}\n")
        
    for i in range(n_resumes):
        print(f"\n=== TESTING RESUME {i+1}/{n_resumes} ===")
        resume_path = f"{test_resumes_dir}dummy_resume_{i+1}.pdf"
        resume_text = parse_resume(resume_path)
        for url in single_page_urls:
            print(f"Filliing application for {url}")
            await fill_application(url, resume_text)

    print(f"Done testing {n_resumes} resumes on single and multi-page applications.")


In [ ]:
cs_urls = ["https://job-boards.greenhouse.io/attentive/jobs/4209296009"]
# cs_urls.append("https://ssctech.wd1.myworkdayjobs.com/ssctechnologies/job/Waltham-MA---10-CityPoint/Software-Engineer-Intern---Co-op_R42466?utm_source=Simplify&ref=Simplify")
# await test_agent_with_dummy_resumes("cs", 1, cs_urls)

In [ ]:
for a in agent_trace_log:
    print(a,"\n")

## What we learned:
* #### Since the LLM does most of the "thinking" and work, most of the issues lie in `fill_application()`.
* #### Checkboxes, text fields, and calendar fields work great. 
* #### However, the agent frequently gets stuck on drop-down fields (picking some option repeatedly), despite the current prompt explicitly telling it skip drop-down fields. We added that prompt so it could at least continue to other fields in the mean time.
* #### The agent struggles with multi-page applications, even though the prompt allows the agent to proceed to the next page so long as the button does not say "Submit" or "Apply" or anything along the lines of "FINAL button for this application." Sometimes the agent goes to the next page, and sometimes it doesn't. We tested the agent with the same site a couple times. 

## Plans before next team report:
* ### Finishing up end-to-end level evals:
  * #### Continue evaluating and improving multi-page application support. As mentioned, the agent inconsistently navigates to the next page, and we want to make this reliable across different job sites (Greenhouse vs. Workday)
  * #### Fix dropdown field handling. It could be the way we prompted the agent in the `fill_application()` function when handling dropdown fields. Right now, we prompted the agent to skip those fields and not interact with them
  * #### Fix the format of agent_reports.csv. Right now, it's showing unknown user, [] for filled/empty fields, and N/A for confidence ratings. Not sure why that's the case since it worked before
* ### Move onto component level evals:
  * #### Just how we did tracing for end-to-end levels, we will apply the same for component levels (agent parsing resume, leveraging LLM calls, format to CSV files, and having user submit the application)

## Team Report #5 (04/27) - Added tracing to our agent with LangSmith

* ### Created a `my_tracer()` function to trace browser-use's thoughts and actions in real time. This is used in addition to our two functions that create agent_reports.csv and application_history.csv
* ### Added Langsmith for some tracing too, although those traces are not visible in this notebook.

<img src="AI_AGENT_WORKFLOW_REPORT_5.png" width="700" />

## Next Steps:
* ### Refactor our code so it's more readable (especially the `fill_application()` function).
* ### Add additional "exit points" in all the functions so we can exit out of them gracefully. Right now `fill_application()` completely kills the kernel if we cancel the operation or close the browser window, which then requires us to rerun all of the cells (inconvenient).
* ### If we find time -> fix the dropdown selection and our agent handle mutlipage job urls.

## Team Report #6 (05/05) - Calculated metrics

* #### Created a `evaluate_field_metrics` function that calculates the following:
    * #### # of total fields in the job application
    * #### % of the fields that has been filled (including the breakdown for each type of fields)
    * #### % and # of each fields that has been filled out
    * #### the correctness rate and amount that has been filed out for each field
    * #### All of these metrics are saved in agent_reports.csv
* #### We also saved the tracing of our agent in agent_traces.csv (format is  ugly right now but all the tracing output where we see the agent thoughts and actions is underneath the `apply_to_job_with_agent`.)


<img src="AI_AGENT_WORKFLOW_REPORT_5.png" width="700" />

## What we learned:
### While repeatedly testing the agent, we realized that our LLM prompt was not detailed enough. The agent would often get stuck on the demographric drop-downs in applications. We realized the agent was often looking for ALL of the possible demographric fields that we mentioned to the agent (e.g. gender, disability, veteran status). We mitigated this issue by modifying the prompt to say something like "use this additional info to fill out demographric questions (any, if they appear)".

### Drop down heavy applications still take longer to process and there is occasional looping, but otherwise, the agent performs as we hoped overall.

## Reflection:
### This was our first time creating an agent, especially toying with browser automation. We learned a lot about the importance of prompt engineering and how to better give an LLM clear, explicit instructions without being too direct (as situations vary).

### Ultimately, the only two issues we weren't able to completely resolve are: proper/fast drop-down interaction and multi-page navigation.

### While it's possible to prompt the agent to click "Next page," we found that it can very quickly become too complicated, especially with our HITL plan. For simplicity, we decided to forgo this idea and stick to refining the agent's performance on single page applications.